# 11 — Louvain k=5: every hospital, every method\n\nPer-hospital macro-F1 for local-only / pooled / FedAvg / Topology-aware, per seed. Output of nb09/nb10 only kept aggregates.

In [ ]:
# 1) Get the code + deps (record versions — deps are unpinned)
!git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
!cd /content/Capstone-amr-fed && git log -1 --oneline
!pip install -q torch_geometric 'flwr[simulation]' networkx
!pip freeze | grep -E '^(torch|torch-geometric|flwr|ray|networkx)==' || true

In [ ]:
# 2) Mount Drive + set ARMD_DIR
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # edit to your folder if different

In [ ]:
# 3) Sanity: data resolves + imports
import sys
sys.path.insert(0, '/content/Capstone-amr-fed/src')
from amr_fed import config
from pathlib import Path
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong — fix cell 2 and re-run.'
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import louvain_split, partition_summary
print('imports OK')

In [ ]:
# 4) Load cohort ONCE — reused by every arm cell below
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame
df = load_cohort_frame()
print('rows:', f'{len(df):,}')

In [ ]:
# 5) Partition per seed: Louvain is seeded, so hospital ids may mean different communities per seed
import pandas as pd
from amr_fed.data_loader import PK
from amr_fed.partition import louvain_split, partition_summary
SEEDS = (42, 43, 44, 45, 46)
for s in SEEDS:
    a = louvain_split(df, n_clients=3, seed=s)
    g, niid = partition_summary(df, a)
    print(f'\n--- seed {s}: {a.nunique()} hospitals ---')
    print(g)


In [ ]:
# 6) Run FedAvg + Topology under the identical protocol; keep EVERY per-hospital number
import os, ray, json
os.environ.pop('AMR_RNG_PROBE', None)
ray.shutdown()
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import louvain_split

KW = dict(seeds=SEEDS, df=df, partition_fn=lambda d, s: louvain_split(d, n_clients=3, seed=s),
          n_clients=3, rounds=8, local_epochs=4, local_only_epochs=32, hidden=128)
fed  = run_multiseed(label='louvain-k5/FedAvg', **KW)
ray.shutdown()
topo = run_multiseed(label='louvain-k5/TOPO', strategy='topology',
                     topo_mode='distinctiveness', topo_temperature=1.0, **KW)
json.dump({'fed': fed, 'topo': topo}, open('/content/drive/MyDrive/amr_louvain_k5_full.json', 'w'), default=str)
print('saved /content/drive/MyDrive/amr_louvain_k5_full.json (raw, all per-hospital values)')


In [ ]:
# 7) Per-hospital table: size | local-only | pooled | FedAvg | Topo   (macro-F1, FedAvg/Topo = best round)
import numpy as np
def f(x): return '  n/a ' if x is None else f'{x:.4f}'
rows = []
for i, s in enumerate(SEEDS):
    rf, rt = fed['runs'][i], topo['runs'][i]
    print(f"\n=== seed {s} ===")
    print(f"{'H':<3}{'n_patients':>11} | {'local':>7} {'pooled':>7} {'FedAvg':>7} {'Topo':>7} | {'Fed-loc':>8} {'Topo-loc':>8}")
    for h in range(len(rf['sizes'])):
        lo, po, fa, tp = rf['local_only_per_client'][h], rf['pooled_per_client'][h], rf['fedavg_per_client'][h], rt['fedavg_per_client'][h]
        print(f"H{h:<2}{rf['sizes'][h]:>11} | {f(lo):>7} {f(po):>7} {f(fa):>7} {f(tp):>7} | {fa-lo:>+8.4f} {tp-lo:>+8.4f}")
        rows.append((s, h, rf['sizes'][h], lo, po, fa, tp))
    print(f"{'avg(weighted)':<14} | {rf['local_only']:.4f} {rf['pooled']:.4f} {rf['fedavg_best']:.4f} {rt['fedavg_best']:.4f}")
    print(f"{'worst':<14} | {min(rf['local_only_per_client']):.4f} {rf['pooled_worst']} {rf['worst_fed']} {rt['worst_fed']}")
    print(f"{'final round':<14} | {'':>7} {'':>7} {rf['fedavg_final']:>7} {rt['fedavg_final']:>7}")
